# Notebook 1: Xác Định Bài Toán & Thiết Kế Dữ Liệu

**Chuyên đề 4: Phân Tích Thị Trường Việc Làm & Gợi Ý Ứng Viên**
**Môn:** Lập trình cho Khoa học Dữ liệu
**Nhóm:** 2 người — 8 buổi
**Ngày:** 2025-07-20

## 1. Mô Tả Bài Toán

### 1.1 Bối cảnh
- Thị trường việc làm IT tại Việt Nam phát triển mạnh, nhưng dữ liệu tuyển dụng phân m tán trên nhiều trang web khác nhau (itviec.com, vietnamworks.com, topdev.vn, careerbuilder.vn).
- Ứng viên khó so sánh lương, kỹ năng yêu cầu, xu hướng theo thành phố/kinh nghiệm.
- Nhà tuyển dụng thiếu benchmark lương thực tế.

### 1.2 Mục tiêu
1. **Cào dữ liệu thực tế** từ ≥2 trang tuyển dụng Việt Nam (ưu tiên itviec, vietnamworks).
2. **Chuẩn hóa** lương (VND/tháng), kỹ năng (synonym map), kinh nghiệm (số năm), vị trí (title mapping).
3. **Phân tích khám phá (EDA)**: Top skills, lương theo kinh nghiệm/thành phố/remote, tỷ lệ lương ẩn, yêu cầu tiếng Anh.
4. **Xây dựng mô hình ML**: Dự đoán lương (LinearRegression, DecisionTree), phân cụm việc làm (K-Means), gợi ý việc làm theo hồ sơ kỹ năng (Content-based cosine similarity).
5. **Đánh giá & báo cáo**: Baseline, error analysis 10+ cases, giới hạn dữ liệu, slide thuyết trình.

### 1.3 Phạm vi
- **Nhóm nghề**: Lập trình / Data (IT jobs) — dễ tìm dữ liệu, thực tế.
- **Địa lý**: TP.HCM, Hà Nội, Đà Nẵng (+ remote).
- **Khoảng thời gian**: Tin đăng 6-12 tháng gần nhất.
- **Số lượng mục tiêu**: ≥1.000 bản ghi sau merge, khuyến khích 2.000-5.000.

## 2. Câu Hỏi Nghiên Cứu (6 câu — F1-F6)

| # | Câu hỏi | Phương pháp | Notebook |
|---|---------|-------------|----------|
| **F1** | Kỹ năng nào được yêu cầu nhiều nhất trong tin tuyển dụng IT? | EDA — Top skills bar chart | 03_eda.ipynb |
| **F2** | Lương thay đổi như thế nào theo kinh nghiệm, thành phố, hình thức làm việc (remote/onsite/hybrid)? | EDA — Boxplot, groupby, pivot heatmap | 03_eda.ipynb |
| **F3** | Yêu cầu tiếng Anh có liên hệ với mức lương không? | EDA — Boxplot có/không English | 03_eda.ipynb |
| **F4** | Vị trí nào thường không công khai lương (hidden salary rate cao nhất)? | EDA — Hidden salary rate by title | 03_eda.ipynb |
| **F5** | Mô hình dự đoán lương sai số bao nhiêu (RMSE, MAE, R²)? | ML — Regression + Error analysis 10+ cases | 04_machine_learning.ipynb |
| **F6** | Top việc làm phù hợp với từng hồ sơ kỹ năng ứng viên? | ML — Content-based recommendation (cosine similarity) | 04_machine_learning.ipynb |

## 3. Thiết Kế Dữ Liệu (Data Design)

### 3.1 Sơ đồ ER (Entity-Relationship)

```
+----------------+       +----------------------+       +--------------+
|   companies    |       |    job_postings      |       |  job_skills  |
+----------------+       +----------------------+       +--------------+
| company_id (PK)|<------| company_id (FK)      |       | job_id (FK)  |
| company_size   |       | job_id (PK)          |------>| skill_name   |
| industry       |       | job_title            |       | skill_group  |
| city           |       | industry             |       | required_level|
| website_url    |       | city                 |       +--------------+
| name_raw       |       | experience_years     |
+----------------+       | education_level      |
                         | job_type             |
                         | contract_type        |
                         | job_level            |
                         | num_hiring           |
                         | remote_option        |
                         | salary_min           |
                         | salary_max           |
                         | salary_hidden        |
                         | working_hours        |
                         | benefits             |
                         | posted_at            |
                         | expired_at           |
                         | description          |
                         | source_site          |
                         | source_url           |
                         | source_html_path     |
                         +----------------------+
```

- **companies** → **job_postings**: 1-N (một công ty đăng nhiều tin)
- **job_postings** → **job_skills**: 1-N (một tin yêu cầu nhiều kỹ năng)

### 3.2 Từ Điển Dữ Liệu (Data Dictionary)

#### Bảng `job_postings`

| Thuộc tính | Kiểu | Mô tả | Ví dụ |
|------------|------|-------|-------|
| job_id | string (PK) | ID duy nhất: `source_site` + hash | `itviec_a1b2c3` |
| job_title | string | Tên vị trí chuẩn hóa | `Backend Developer` |
| company_id | string (FK) | ID công ty | `comp_itviec_001` |
| industry | string | Ngành ngành | `Information Technology` |
| city | string | Thành phố làm việc | `HCMC` |
| experience_years | float | Số năm kinh nghiệm (chuẩn hóa) | `3.0` |
| experience_bin | category | Nhóm kinh nghiệm | `mid` (entry/junior/mid/senior/lead) |
| education_level | category | Trình độ học vấn | `Bachelor`, `Master`, `Not specified` |
| job_type | category | Loại hình việc làm | `Full-time`, `Part-time`, `Contract`, `Intern` |
| contract_type | category | Loại hợp đồng | `CDH (Indefinite)`, `Temporary`, `Freelance`, `Not specified` |
| job_level | category | Cấp bậc vị trí | `Employee`, `Team Lead`, `Manager`, `Director`, `Not specified` |
| num_hiring | int | Số lượng cần tuyển | `5` |
| remote_option | category | Tùy chọn remote | `On-site`, `Hybrid`, `Remote`, `Not specified` |
| salary_min | float | Lương tối thiểu (triệu VND/tháng) | `15.0` |
| salary_max | float | Lương tối đa (triệu VND/tháng) | `25.0` |
| salary_mid | float | Lương trung bình (min+max)/2 | `20.0` |
| salary_hidden | bool | Có ẩn lương không | `True`/`False` |
| working_hours | string | Giờ làm việc | `Hành chính (8h-17h)`, `Flexible`, `Theo ca` |
| benefits | text | Phúc lợi | `Bảo hiểm, ăn trưa, du lịch hằng năm` |
| posted_at | datetime | Ngày đăng tin | `2025-06-15` |
| expired_at | datetime | Hạn nộp hồ sơ | `2025-07-15` |
| description | text | Mô tả chi tiết tin tuyển dụng | `...` |
| source_site | string | Nguồn cào dữ liệu | `itviec`, `vietnamworks`, `topdev` |
| source_url | string | URL gốc tin tuyển dụng | `https://itviec.com/jobs/...` |
| source_html_path | string | Đường dẫn file HTML raw | `data/raw/html/itviec_a1b2c3.html` |

<small>🆕 Field mới: `contract_type`, `job_level`, `num_hiring`, `working_hours`, `benefits`, `expired_at`</small>

#### Bảng `job_skills`

| Thuộc tính | Kiểu | Mô tả | Ví dụ |
|------------|------|-------|-------|
| job_id | string (FK) | FK đến job_postings | `itviec_a1b2c3` |
| skill_name | string | Tên kỹ năng đã chuẩn hóa | `Python` |
| original_skill_name | string | Tên kỹ năng gốc trước chuẩn hóa | `Python3`, `python` |
| skill_group | category | Nhóm kỹ năng | `Programming Language`, `Database`, `Framework`, `Tool`, `Cloud` |
| required_level | category | Mức độ yêu cầu | `Required`, `Nice to have`, `Not specified` |

#### Bảng `companies`

| Thuộc tính | Kiểu | Mô tả | Ví dụ |
|------------|------|-------|-------|
| company_id | string (PK) | ID duy nhất công ty | `comp_itviec_001` |
| company_name | string | Tên công ty | `FPT Software` |
| company_name_raw | string | Tên gốc trước chuẩn hóa | `FPT Software Viet Nam` |
| company_size | category | Quy mô | `Startup (1-50)`, `SME (51-200)`, `Large (201-1000)`, `Enterprise (1000+)` |
| industry | string | Ngành ngành | `Information Technology` |
| city | string | Thành phố trụ sở | `HCMC` |
| website_url | string | Website công ty | `https://fpt.com.vn` |
| source_site | string | Nguồn cào | `itviec` |

<small>🆕 Field mới: `company_name_raw`, `website_url`</small>

## 4. Kiến Trúc OOP (Domain Classes)

### 4.1 Tổng quan các class

| Class | File | Chịu trách nhiệm |
|-------|------|------------------|
| `JobPosting` | `src/domain/job_posting.py` | Domain entity cho 1 tin tuyển dụng |
| `Skill` | `src/domain/skill.py` | Domain entity cho 1 kỹ năng đã chuẩn hóa |
| `Company` | `src/domain/company.py` | Domain entity cho công ty |
| `JobDataManager` | `src/data/data_manager.py` | Data layer: đọc/ghi/merge/log dữ liệu |
| `RecommendationEngine` | `src/ml/recommendation.py` | ML layer: content-based recommendation |

### 4.2 JobPosting Dataclass

In [1]:
# Preview JobPosting class structure
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional, List

@dataclass
class JobPosting:
    """Domain entity: một tin tuyển dụng."""
    job_id: str
    job_title: str
    company_id: str
    industry: str
    city: str
    experience_years: float
    experience_bin: str  # entry, junior, mid, senior, lead
    education_level: str
    job_type: str
    remote_option: str
    salary_min: Optional[float] = None
    salary_max: Optional[float] = None
    salary_mid: Optional[float] = None
    salary_hidden: bool = False
    posted_at: Optional[datetime] = None
    description: str = ""
    source_site: str = ""
    source_url: str = ""
    source_html_path: str = ""
    skills: List['Skill'] = field(default_factory=list)

    @property
    def has_salary(self) -> bool:
        return self.salary_min is not None or self.salary_max is not None

    def to_dict(self) -> dict:
        d = {k: v for k, v in self.__dict__.items() if k != 'skills'}
        d['posted_at'] = self.posted_at.isoformat() if self.posted_at else None
        d['skills'] = [s.to_dict() for s in self.skills]
        return d

print("JobPosting class structure defined.")
print([f for f in JobPosting.__dataclass_fields__.keys()])


JobPosting class structure defined.
['job_id', 'job_title', 'company_id', 'industry', 'city', 'experience_years', 'experience_bin', 'education_level', 'job_type', 'remote_option', 'salary_min', 'salary_max', 'salary_mid', 'salary_hidden', 'posted_at', 'description', 'source_site', 'source_url', 'source_html_path', 'skills']


### 4.3 Skill Dataclass

In [2]:
@dataclass
class Skill:
    """Domain entity: một kỹ năng đã chuẩn hóa."""
    skill_name: str           # Tên chuẩn hóa (canonical)
    original_name: str        # Tên gốc trước khi chuẩn hóa
    skill_group: str          # programming_language, framework, database, tool, cloud, language, other
    required_level: str       # required, nice_to_have, not_specified
    job_id: str = ""          # FK đến job_postings

    def to_dict(self) -> dict:
        return self.__dict__.copy()

print("Skill class structure defined.")
print([f for f in Skill.__dataclass_fields__.keys()])


Skill class structure defined.
['skill_name', 'original_name', 'skill_group', 'required_level', 'job_id']


### 4.4 Company Dataclass

In [3]:
@dataclass
class Company:
    """Domain entity: công ty."""
    company_id: str
    company_name: str
    company_size: str
    industry: str
    city: str
    source_site: str = ""

    def to_dict(self) -> dict:
        return self.__dict__.copy()

print("Company class structure defined.")


Company class structure defined.


## 5. Tu dien du lieu (Data Dictionary)

| Column | Kieu du lieu | Mo ta | Gia tri vi du |
|--------|-------------|-------|---------------|
| job_id | string | Ma dinh danh | fallback_000000 |
| job_title | string | Ten vi tri tuyen dung | Machine Learning Engineer |
| company_id | string | Ma cong ty (FK -> companies) | comp_5a84e068 |
| industry | string | Nganh nghe | IT |
| city | string | Thanh pho | HCMC, Hanoi, Da Nang |
| experience_years | float | So nam kinh nghiem | 3.5 |
| education_level | string | Trinh do hoc van | Bachelor, Master, PhD |
| job_type | string | Loai hinh cong viec | Full-time, Part-time, Contract |
| remote_option | string | Hinh thuc lam viec | On-site, Hybrid, Remote |
| salary_min | float | Luong toi thieu (trieu VND/thang) | 10.0 |
| salary_max | float | Luong toi da (trieu VND/thang) | 20.0 |
| salary_mid | float | Luong trung binh | 15.0 |
| salary_hidden | bool | Luong khong cong khai | True/False |
| has_english | bool | Co yeu cau tieng Anh | True/False |
| posted_at | string | Ngay dang tuyen | 2025-06-01 |
| skills | list[string] | Danh sach ky nang yeu cau | ["Python", "SQL"] |
| skill_groups | list[string] | Nhom ky nang | ["Programming Language"] |
| experience_bin | string | Phan nhom kinh nghiem | entry, junior, mid, senior, lead |
| company_name | string | Ten cong ty | FPT Software |
| company_size | string | Quy mo cong ty | Large (201-1000) |
| source_site | string | Nguon du lieu | itviec, vietnamworks, fallback |


In [ ]:
import sys; sys.path.append(".."); import pandas as pd
from src.data.data_manager import JobDataManager
from src.domain.job_posting import JobPosting
from src.domain.skill import Skill
from src.domain.company import Company

# Load du lieu da xu ly (combined.csv tu crawl v2)
dm = JobDataManager(processed_dir="../data/processed")
combined_path = dm.processed_dir / "combined.csv"
if not combined_path.exists():
    raise FileNotFoundError(f"{combined_path} not found. Run 'python crawl.py' first.")
df = pd.read_csv(combined_path, encoding="utf-8-sig")
print(f"Loaded: {len(df)} rows, {df.shape[1]} cols")
print(f"Cities (top 10): {df['city'].value_counts().head(10).to_dict()}")
print(f"Skills coverage: {df['skills'].notna().sum()}/{len(df)}")
print(f"Salary_mid NaN: {df['salary_mid'].isna().sum()}")
print()
print("=== OOP Domain Classes Demo ===")
# JobPosting demo
jp = JobPosting(
    job_title="Data Scientist", company_id="comp_demo", city="HCMC",
    source_url="https://example.com", source_site="demo",
    experience_years=3.5, education_level="Master",
    salary_min=20.0, salary_max=40.0,
)
print(f"JobPosting: {jp.job_title} - {jp.city} (salary_mid={jp.salary_mid}M)")
# Skill demo
sk = Skill(skill_name="Python", original_name="py", skill_group="Programming Language")
print(f"Skill: {sk.skill_name} (group: {sk.skill_group})")
# Company demo
co = Company(company_id="comp_demo", company_name="Tech Corp", company_size="Enterprise")
print(f"Company: {co.company_name} (size: {co.company_size})")

## 5. Kế Hoạch Triển Khai Tiếp Theo

| Bước | Nội dung | File chính |
|------|----------|------------|
| 1 | ✅ Setup project structure, requirements, README | (done) |
| 2 | ✅ Notebook 1: Problem definition, RQs, ER, Data dict, OOP | `01_problem_and_data.ipynb` |
| 3 | **Buổi 2**: Viết code OOP classes vào `src/domain/*.py` | `src/domain/` |
| 4 | **Buổi 3**: Crawl dữ liệu thật (`collector.py`, `salary_parser.py`) | `src/data/` |
| 5 | **Buổi 4**: Cleaning pipeline (skill, experience, dedup) | `src/cleaning/` |
| 6 | **Buổi 5**: Merge dữ liệu, 5 bảng Groupby/Pivot | `02_collection_and_cleaning.ipynb` |
| 7 | **Buổi 6**: EDA — 8+ biểu đồ, trả lời F1-F4 | `03_eda.ipynb` |
| 8 | **Buổi 7**: ML — Baseline, LinearRegression, DecisionTree, Error analysis | `04_machine_learning.ipynb` |
| 9 | **Buổi 8**: Clustering + Recommendation + Báo cáo hoàn chỉnh | `04_machine_learning.ipynb` + `reports/` |

## 6. Phân Công Buổi 1 (K1)

| Thành viên | Nhiệm vụ |
|------------|----------|
| Member 1 | Tạo repo, cấu trúc thư mục, requirements.txt, .gitignore, README.md |
| Member 2 | Viết Notebook 1: bài toán, 6 câu hỏi nghiên cứu, ER diagram, data dictionary |
| Cả 2 | Review cùng nhau, thống nhất SKILL_SYNONYM_MAP (35+ entry), commit lên git |

---

**Sẵn sàng cho Buổi 2:** Viết code OOP classes vào `src/domain/`.